In [1]:
import os
import os.path as op
import mne
import nibabel as nib
import numpy as np

In [2]:
# Setup paths
path = '/Users/immlab/Desktop/IMM-Lab'
meg_path = op.join(path, 'MEG')
mri_path = op.join(path, 'MRI')

# Load source space
src_path = op.join(mri_path, 'fsaverage', 'bem', 'fsaverage-mixed-src.fif')
src = mne.read_source_spaces(src_path, verbose=False)

In [ ]:
def process_and_convert_stc_to_nifti(input_folder, output_folder, src, resample_freq=None):
    """
    Convert all STC files in a folder to NIfTI format.
    For each STC, finds peak activation time, extracts ±20ms around peak (41 time points),
    and saves two NIfTI files: one with all 41 volumes, one with the average volume.
    
    Parameters:
    - input_folder: Path to folder containing STC files
    - output_folder: Path to save NIfTI files (will be created if doesn't exist)
    - src: MNE source space object
    - resample_freq: Optional frequency to resample
    """
    
    os.makedirs(output_folder, exist_ok=True)
    
    stc_files = [f for f in os.listdir(input_folder) if f.endswith('-stc.h5')]
    
    if not stc_files:
        print(f"No STC files found in {input_folder}")
        return
    
    print(f"Found {len(stc_files)} STC files in {input_folder}")
    print(f"Output folder: {output_folder}")
    
    for stc_file in stc_files:
        try:
            stc_path = op.join(input_folder, stc_file)
            stc = mne.read_source_estimate(stc_path)
            
            print(f"\nProcessing {stc_file}:")
            print(f"  Original: {1/stc.tstep:.0f} Hz, {len(stc.times)} time points, {stc.times[0]:.3f} to {stc.times[-1]:.3f} s")
            
            if resample_freq is not None:
                print(f"  Resampling from {1/stc.tstep:.0f} Hz to {resample_freq} Hz")
                stc = stc.copy().resample(resample_freq)
            
            global_activation = np.sum(np.abs(stc.data), axis=0)
            peak_time_idx = np.argmax(global_activation)
            peak_time = stc.times[peak_time_idx]
            peak_value = global_activation[peak_time_idx]
            
            print(f"  Peak activation: {peak_value:.2e} at t = {peak_time:.3f} s (index {peak_time_idx})")
            
            time_window_ms = 20  # milliseconds
            time_window_s = time_window_ms / 1000  # convert to seconds
            
            samples_per_20ms = int(time_window_s / stc.tstep)
            
            start_idx = max(0, peak_time_idx - samples_per_20ms)
            end_idx = min(len(stc.times), peak_time_idx + samples_per_20ms + 1)
            
            actual_window = end_idx - start_idx
            if actual_window < 41 and len(stc.times) >= 41:
                deficit = 41 - actual_window
                expand_left = deficit // 2
                expand_right = deficit - expand_left
                
                new_start = max(0, start_idx - expand_left)
                new_end = min(len(stc.times), end_idx + expand_right)
                
                start_idx, end_idx = new_start, new_end
            
            stc_windowed = stc.copy().crop(stc.times[start_idx], stc.times[end_idx-1])
            
            print(f"  Extracted window: {len(stc_windowed.times)} time points from {stc_windowed.times[0]:.3f} to {stc_windowed.times[-1]:.3f} s")
            print(f"  Window duration: {(stc_windowed.times[-1] - stc_windowed.times[0])*1000:.1f} ms")
            
            vol_stc = stc_windowed.volume()
            
            nii_4d = vol_stc.as_volume(src, mri_resolution=True)
            
            nii_data_avg = np.mean(nii_4d.get_fdata(), axis=-1)
            
            nii_3d = nib.Nifti1Image(nii_data_avg, nii_4d.affine, nii_4d.header)
            
            base_name = stc_file.replace('-stc.h5', '')
            suffix = f"_resampled_{resample_freq}Hz" if resample_freq else ""
            
            nii_4d_filename = f"{base_name}{suffix}_peak_window_4D.nii.gz"
            nii_4d_path = op.join(output_folder, nii_4d_filename)
            
            nii_3d_filename = f"{base_name}{suffix}_peak_window_avg.nii.gz"
            nii_3d_path = op.join(output_folder, nii_3d_filename)
            
            nib.save(nii_4d, nii_4d_path)
            nib.save(nii_3d, nii_3d_path)
            
        except Exception as e:
            print(f"Error processing {stc_file}: {str(e)}")
    
    print(f"Files saved to: {output_folder}")

In [ ]:
# function to extract at a time point
def extract_timepoint_nifti(input_folder, output_folder, src, time_point, resample_freq=None):
    """
    Convert all STC files in a folder to NIfTI format.
    For each STC, extracts a single time point and saves as a NIfTI file.
    
    Parameters:
    - input_folder: Path to folder containing STC files
    - output_folder: Path to save NIfTI files (will be created if doesn't exist)
    - src: MNE source space object
    - time_point: Time point in seconds to extract
    - resample_freq: Optional frequency to resample
    """
    
    os.makedirs(output_folder, exist_ok=True)
    
    stc_files = [f for f in os.listdir(input_folder) if f.endswith('-stc.h5')]
    
    if not stc_files:
        print(f"No STC files found in {input_folder}")
        return
    
    print(f"Found {len(stc_files)} STC files in {input_folder}")
    print(f"Output folder: {output_folder}")
    
    total_4d_size = 0
    total_3d_size = 0
    
    for stc_file in stc_files:
        try:
            stc_path = op.join(input_folder, stc_file)
            stc = mne.read_source_estimate(stc_path)
            
            print(f"\nProcessing {stc_file}:")
            print(f"  Original: {1/stc.tstep:.0f} Hz, {len(stc.times)} time points, {stc.times[0]:.3f} to {stc.times[-1]:.3f} s")
            
            if resample_freq is not None:
                print(f"  Resampling from {1/stc.tstep:.0f} Hz to {resample_freq} Hz")
                stc = stc.copy().resample(resample_freq)
            
            if time_point < stc.times[0] or time_point > stc.times[-1]:
                print(f"  Time point {time_point:.3f} s is out of bounds ({stc.times[0]:.3f} to {stc.times[-1]:.3f} s). Skipping.")
                continue
            
            time_idx = np.argmin(np.abs(stc.times - time_point))
            actual_time = stc.times[time_idx]
            
            print(f"  Extracting time point at t = {actual_time:.3f} s (index {time_idx})")
            stc_timepoint = stc.copy().crop(actual_time, actual_time)
            vol_stc = stc_timepoint.volume()
            nii_3d = vol_stc.as_volume(src, mri_resolution=True)
            nii_data = nii_3d.get_fdata()
            nii_3d = nib.Nifti1Image(nii_data, nii_3d.affine, nii_3d.header)
            base_name = stc_file.replace('-stc.h5', '')
            suffix = f"_resampled_{resample_freq}Hz" if resample_freq else ""
            nii_3d_filename = f"{base_name}{suffix}_time_{actual_time*1000:.0f}ms.nii.gz"
            nii_3d_path = op.join(output_folder, nii_3d_filename)
            nib.save(nii_3d, nii_3d_path)
        except Exception as e:
            print(f"Error processing {stc_file}: {str(e)}")
            continue
    print(f"Files saved to: {output_folder}")


In [6]:
adult_input_folder = op.join(meg_path, 'adult_morphed_averaged')
adult_output_folder = op.join(meg_path, 'adult_morphed_averaged', 'nifti_files')

process_and_convert_stc_to_nifti(adult_input_folder, adult_output_folder, src)

Found 9 STC files in /Users/immlab/Desktop/IMM-Lab/MEG/adult_morphed_averaged
Output folder: /Users/immlab/Desktop/IMM-Lab/MEG/adult_morphed_averaged/nifti_files

Processing VML_adult_morphed-averaged_DS_left no-rotation-stc.h5:
  Original: 1000 Hz, 1701 time points, -0.500 to 1.200 s
  Peak activation: 5.33e+02 at t = 0.249 s (index 749)
  Extracted window: 41 time points from 0.229 to 0.269 s
  Window duration: 40.0 ms

Processing VML_adult_morphed-averaged_DS_run 4-stc.h5:
  Original: 1000 Hz, 1701 time points, -0.500 to 1.200 s
  Peak activation: 4.12e+02 at t = 0.675 s (index 1175)
  Extracted window: 41 time points from 0.655 to 0.695 s
  Window duration: 40.0 ms

Processing VML_adult_morphed-averaged_DS_run 5 -stc.h5:
  Original: 1000 Hz, 1701 time points, -0.500 to 1.200 s
  Peak activation: 4.29e+02 at t = -0.083 s (index 417)
  Extracted window: 41 time points from -0.103 to -0.063 s
  Window duration: 40.0 ms

Processing VML_adult_morphed-averaged_DS_right no-rotation-stc.h5

In [7]:
child_input_folder = op.join(meg_path, 'child_morphed_averaged')
child_output_folder = op.join(meg_path, 'child_morphed_averaged', 'nifti_files')

process_and_convert_stc_to_nifti(child_input_folder, child_output_folder, src)

Found 9 STC files in /Users/immlab/Desktop/IMM-Lab/MEG/child_morphed_averaged
Output folder: /Users/immlab/Desktop/IMM-Lab/MEG/child_morphed_averaged/nifti_files

Processing VML_child_morphed-averaged_DS_right rotation-stc.h5:
  Original: 1000 Hz, 1701 time points, -0.500 to 1.200 s
  Peak activation: 4.59e+02 at t = 0.307 s (index 807)
  Extracted window: 41 time points from 0.287 to 0.327 s
  Window duration: 40.0 ms

Processing VML_child_morphed-averaged_DS_right no-rotation-stc.h5:
  Original: 1000 Hz, 1701 time points, -0.500 to 1.200 s
  Peak activation: 5.97e+02 at t = 0.341 s (index 841)
  Extracted window: 41 time points from 0.321 to 0.361 s
  Window duration: 40.0 ms

Processing VML_child_morphed-averaged_DS_run 2 -stc.h5:
  Original: 1000 Hz, 1701 time points, -0.500 to 1.200 s
  Peak activation: 4.93e+02 at t = 0.306 s (index 806)
  Extracted window: 41 time points from 0.286 to 0.326 s
  Window duration: 40.0 ms

Processing VML_child_morphed-averaged_DS_run 1 -stc.h5:
  O